# PulseAI Agent Core on Colibri (Kaggle GPU)

This notebook is a thin, reproducible launcher. It clones the active PulseAI test branch, then delegates all resource gates, the pinned Colibri base plus its immutable upstream partial-residency patch, model-revision resolution, endpoint probes, the bounded Agent Core turn, and evidence validation to `benchmarks.colibri_agent_core.kaggle_runner`.

**Selected model:** Qwen3.6-35B-A3B, Colibri int4-gs64. The first run configures 65,536 tokens and probes approximately 16,384 filler tokens. The 262,144-token architecture limit is not reported as a successful run unless separately exercised and evidenced. Model weights live under `/kaggle/temp` and are never included in kernel outputs; only bounded receipts, logs, and Agent Core evidence are written under `/kaggle/working`.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

PULSEAI_REPOSITORY = "https://github.com/SriAkhilSJ/PulseAIRepo.git"
PULSEAI_REF = os.environ.get("PULSEAI_REF", "arena/a7c2c868-pulseairepo")
PULSEAI_DIR = Path("/kaggle/temp/PulseAIRepo")
OUTPUT_ROOT = Path("/kaggle/working/colibri-agent-core-results")

if PULSEAI_DIR.exists():
    raise RuntimeError(f"Refusing to overwrite existing checkout: {PULSEAI_DIR}")
PULSEAI_DIR.parent.mkdir(parents=True, exist_ok=True)
clone_env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1")
subprocess.run(
    [
        "git",
        "clone",
        "--no-tags",
        "--single-branch",
        "--branch",
        PULSEAI_REF,
        PULSEAI_REPOSITORY,
        str(PULSEAI_DIR),
    ],
    check=True,
    env=clone_env,
)
pulseai_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PULSEAI_DIR, text=True
).strip()
print(f"PulseAI ref {PULSEAI_REF} resolved to {pulseai_commit}", flush=True)

command = [
    sys.executable,
    "-m",
    "benchmarks.colibri_agent_core.kaggle_runner",
    "--runtime-root",
    "/kaggle/temp/pulseai-colibri",
    "--output-root",
    str(OUTPUT_ROOT),
    "--context-window",
    "65536",
    "--long-context-tokens",
    "16384",
    "--server-start-timeout",
    "1800",
    "--probe-timeout",
    "1800",
    "--agent-timeout",
    "3600",
]
run_env = dict(os.environ, PYTHONUNBUFFERED="1")
completed = subprocess.run(command, cwd=PULSEAI_DIR, env=run_env, check=False)
receipts = sorted(OUTPUT_ROOT.glob("run-*/run_receipt.json"))
if receipts:
    receipt = json.loads(receipts[-1].read_text(encoding="utf-8"))
    print(json.dumps(receipt, indent=2, sort_keys=True), flush=True)
else:
    print("No run receipt was produced.", flush=True)
if completed.returncode:
    raise RuntimeError(
        f"Colibri Agent Core run failed with exit code {completed.returncode}; "
        "download the result bundle for the bounded failure receipts."
    )